# FOV analysis scheduler

Run this notebook **during acquisition** in its own JupyterLab tab or kernel.

Analysis runs **continuously** — during both acquisition and fluidics — and
processes FOVs **in parallel** across a pool of worker processes. For each new
image file it creates:
- Thumbnails (PNG) for each frame
- Per-frame intensity stats (CSV)
- Intensity histograms (`.npz`)

Each FOV is marked complete with a zero-byte sentinel so that
`02_round_scheduler.ipynb` knows when to start building mosaics.

**Analysis modes** (`ANALYSIS_MODE` below)

* **`"same_drive"`** (mode B) — analyse straight from the acquisition drive
  (`data/`). Simplest; analysis I/O shares the microscope drive (possible
  contention on slow HDDs).
* **`"mirror_drive"`** (mode A) — during fluidics, incrementally mirror `data/`
  to a second drive (`ANALYSIS_SOURCE_DIR`) and analyse from that mirror, so
  analysis reads never compete with the microscope's writes during acquisition.
* **`"round_robin_drives"`** — for experiments whose `round_info.csv` spreads hyb
  rounds round-robin across several physical drives (see prepare_imaging/03's
  `DATA_DRIVES`). Reads directly from every drive referenced in `round_info.csv`
  and skips only the round HAL is actively writing right now, so completed rounds
  on other drives analyse immediately with no mirroring step.

**Parallelism** — `N_ANALYSIS_WORKERS` worker processes each handle one FOV
(read once, run all analyses). Default is `cpu_count − 2`. Each worker holds one
image stack (~200 MB) in memory, so lower this if memory is tight.

The optional NAS transfer (`TRANSFER_DEST`) is independent and still runs only
during the fluidics window.

## 1 — Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config   import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.progress        import ProgressTracker
from MERci.state           import ExperimentStateMonitor
from MERci.scheduler       import FOVScheduler, RoundScheduler
from MERci.visualization   import display_mosaic

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Experiment parameters

Edit the cells below to match your experiment.

In [ ]:
SAMPLE_NAME = SAMPLE_DIR.name

# ── Fluidics type ──────────────────────────────────────────────────────
# "adaptor"  →  t_max = 100 min  (adaptor-based fluidics)
# "direct"   →  t_max =  50 min  (direct-readout fluidics)
FLUIDICS_TYPE = "adaptor"

# ── Image file format (must match what HAL writes) ─────────────────────
IMAGE_SUFFIX = ".zarr"   # options: ".zarr", ".dax", ".tiff"

# ── Analysis mode ──────────────────────────────────────────────────────
# "same_drive"   (B): analyse from data/ on the acquisition drive (simplest).
# "mirror_drive" (A): mirror data/ to a 2nd drive during fluidics and analyse
#                     from there, so analysis I/O never competes with acquisition.
# "round_robin_drives": read directly from every drive round_info.csv spreads hyb
#                     rounds across (see prepare_imaging/03's DATA_DRIVES); skip only
#                     the round currently being written.
ANALYSIS_MODE       = "same_drive"
# Only used when ANALYSIS_MODE == "mirror_drive": a directory ON A SECOND DRIVE
# to mirror data into and analyse from, e.g. r"E:\merci_mirror\LT027\data"
ANALYSIS_SOURCE_DIR = None

# ── Parallelism ────────────────────────────────────────────────────────
# Number of FOV worker processes. None → cpu_count - 2. Each worker holds one
# image stack (~200 MB) in RAM, so lower this if memory is tight. Set to 1 to
# run serially in-process (easiest to debug).
N_ANALYSIS_WORKERS = None

# ── FOV subset (optional) ──────────────────────────────────────────────
# Set to a list of FOV ids to process only a subset, e.g. [0, 1, 2, 3]
# Leave as None to process all FOVs
FOV_SUBSET = None

# ── Thumbnail frames (optional) ────────────────────────────────────────
# Set to a list of frame indices to limit thumbnail creation, e.g. [2, 3, 4]
# Leave as None to thumbnail all frames
THUMBNAIL_FRAMES = None

# ── Data transfer (optional, independent of ANALYSIS_MODE) ─────────────
# Set to a network path to copy completed round data during the fluidics
# window.  Transfer starts only when ≥ TRANSFER_MIN_TIME seconds remain.
# Leave as None to skip transfer.
TRANSFER_DEST     = None   # e.g. r"\\NAS\experiments\LT027"
TRANSFER_MIN_TIME = 600    # seconds (10 min)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"Fluidics type  : {FLUIDICS_TYPE}")
print(f"Image suffix   : {IMAGE_SUFFIX}")
print(f"Analysis mode  : {ANALYSIS_MODE}")
print(f"Analysis source: {ANALYSIS_SOURCE_DIR}")
print(f"Workers        : {N_ANALYSIS_WORKERS}")
print(f"FOV subset     : {FOV_SUBSET}")
print(f"Thumb frames   : {THUMBNAIL_FRAMES}")
print(f"Transfer dest  : {TRANSFER_DEST}")

In [ ]:
config = ExperimentConfig(
    data_dir            = SAMPLE_DIR / "data",
    metadata_dir        = SAMPLE_DIR / "metadata",
    analysis_dir        = SAMPLE_DIR / "analysis",
    settings_dir        = SAMPLE_DIR / "settings",
    round_info_csv      = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt       = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    fluidics_type       = FLUIDICS_TYPE,
    image_suffix        = IMAGE_SUFFIX,
    analysis_mode       = ANALYSIS_MODE,
    analysis_source_dir = ANALYSIS_SOURCE_DIR,
    n_analysis_workers  = N_ANALYSIS_WORKERS,
    fov_subset          = FOV_SUBSET,
    thumbnail_frames    = THUMBNAIL_FRAMES,
    transfer_dest       = TRANSFER_DEST,
    transfer_min_time   = TRANSFER_MIN_TIME,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)
monitor = ExperimentStateMonitor(config)

print(f"Rounds         : {meta.n_rounds}")
print(f"FOVs           : {meta.n_fovs}")
print(f"Analysis mode  : {config.analysis_mode}")
print(f"Analysis source: {config.analysis_data_dir}")
print(f"Workers        : {config.resolved_n_workers}")
print(f"Analysis dir   : {config.analysis_dir}")
if config.transfer_dest:
    print(f"Transfer dest  : {config.transfer_dest}  (min time: {config.transfer_min_time:.0f} s)")

## 3 — Check current progress

Run this cell at any time to see how many FOVs and rounds have been processed.

In [ ]:
summary = tracker.summary(meta)
print(f"FOVs  done : {summary['files_fov_done']} / {summary['files_total']}")
print(f"Rounds done: {summary['rounds_done']} / {summary['rounds_total']}")

## 4 — FOV scheduler

**Run this cell in one notebook tab** (or kernel) and leave it running.

On every tick it:
- (mirror mode only) refreshes the second-drive mirror while the microscope is idle
- Discovers pending image files and analyses them **continuously**, **in parallel** across `N_ANALYSIS_WORKERS` worker processes (each reads one file and creates its thumbnails, per-frame stats, and intensity histograms)
- Marks each FOV complete with a zero-byte sentinel when all outputs are written

Interrupt the kernel (`■` button) to stop the loop gracefully (the worker pool is shut down on exit).

> **Windows note:** parallel workers use `ProcessPoolExecutor`. If you ever see worker-startup errors, set `N_ANALYSIS_WORKERS = 1` to run serially in-process.

In [ ]:
def show_phase(phase):
    from IPython.display import clear_output
    clear_output(wait=True)
    tsi = f"{phase.time_since_imaging:.0f} s" if phase.time_since_imaging is not None else "n/a"
    print(f"Phase         : {phase.phase_str}   (analysis runs continuously)")
    print(f"Time since img: {tsi}")
    print(f"Mode / workers: {config.analysis_mode} / {config.resolved_n_workers}")
    smry = tracker.summary(meta)
    print(f"FOVs done     : {smry['files_fov_done']} / {smry['files_total']}")

FOVScheduler(config, meta, tracker, monitor).run_loop(on_phase_update=show_phase)